# Embeddings - Comprehensive Implementation Guide

This notebook covers:
- **Basic Implementation**: Simple, educational version
- **Advanced Implementation**: Production-ready patterns
- **Real-World Examples**: How companies use this in production
- **Integration**: Using popular libraries

Source: `llm/concepts/embeddings.md`

## Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np

print("Libraries loaded successfully")

## 1. Basic Implementation

Simple, educational version to understand core concepts.

In [ ]:
# Basic Embedding from Scratch
import torch
import torch.nn as nn

class SimpleEmbedding(nn.Module):
    """Basic embedding: transform text tokens to vectors"""

    def __init__(self, vocab_size=10000, embedding_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

    def forward(self, input_ids):
        return self.embedding(input_ids)

# Test
embed = SimpleEmbedding(vocab_size=10000, embedding_dim=128)
input_ids = torch.randint(0, 10000, (2, 10))  # batch=2, seq_len=10
embeddings = embed(input_ids)
print(f"Input shape: {input_ids.shape}")
print(f"Embedding shape: {embeddings.shape}")  # (2, 10, 128)

## 2. Advanced Implementation

Production-ready patterns with optimization and error handling.

In [ ]:
# Advanced: Sentence Embeddings with Pooling
import torch
import torch.nn as nn
import torch.nn.functional as F

class SentenceEmbedding(nn.Module):
    """Convert sentences to fixed-size embeddings"""

    def __init__(self, hidden_size=768, embedding_dim=384):
        super().__init__()
        self.transform = nn.Linear(hidden_size, embedding_dim)

    def forward(self, hidden_states, attention_mask=None):
        # Mean pooling: average over sequence length
        if attention_mask is not None:
            attention_mask = attention_mask.unsqueeze(-1).float()
            hidden_states = hidden_states * attention_mask
            sum_embeddings = torch.sum(hidden_states, dim=1)
            sum_mask = torch.sum(attention_mask, dim=1)
            embeddings = sum_embeddings / sum_mask
        else:
            embeddings = torch.mean(hidden_states, dim=1)

        # Project to embedding dim
        embeddings = self.transform(embeddings)
        # Normalize
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return embeddings

# Usage
pooler = SentenceEmbedding(hidden_size=768, embedding_dim=384)
hidden = torch.randn(2, 10, 768)  # 2 sentences, 10 tokens each
embeddings = pooler(hidden)
print(f"Sentence embeddings shape: {embeddings.shape}")  # (2, 384)

# Compute similarity
similarity = torch.mm(embeddings, embeddings.t())
print(f"Similarity matrix:\n{similarity}")

## Real-World: Sentencebert

How this is used in production systems.

In [ ]:
# Real-World: Using Sentence-BERT
from sentence_transformers import SentenceTransformer, util
import numpy as np

# Load pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode sentences
sentences = [
    "The cat is on the mat",
    "A feline rests on a rug",
    "Python is a programming language",
    "Java is also a programming language"
]

embeddings = model.encode(sentences, convert_to_tensor=True)

# Compute similarity
similarities = util.pytorch_cos_sim(embeddings, embeddings)

# Find most similar pairs
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        sim = similarities[i][j].item()
        print(f"{sentences[i]} <-> {sentences[j]}: {sim:.3f}")

## Real-World: Production

How this is used in production systems.

In [ ]:
# Real-World: Production Embedding Pipeline
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

class EmbeddingService:
    """Production embedding service for semantic search"""

    def __init__(self, model_name='all-MiniLM-L6-v2', batch_size=32):
        self.model = SentenceTransformer(model_name)
        self.batch_size = batch_size
        self.document_cache = {}

    def embed_documents(self, documents):
        """Embed documents and cache them"""
        embeddings = self.model.encode(documents, batch_size=self.batch_size)
        for doc, emb in zip(documents, embeddings):
            self.document_cache[hash(doc)] = emb
        return embeddings

    def search(self, query, documents, top_k=5):
        """Find top-k most similar documents"""
        query_emb = self.model.encode([query])[0]
        doc_embs = self.embed_documents(documents)

        similarities = cosine_similarity([query_emb], doc_embs)[0]
        top_indices = np.argsort(similarities)[::-1][:top_k]

        results = [
            {"doc": documents[i], "score": similarities[i]}
            for i in top_indices
        ]
        return results

# Usage
service = EmbeddingService()
docs = [
    "Machine learning uses algorithms to learn from data",
    "Deep learning uses neural networks",
    "Python is popular for ML"
]
results = service.search("What is machine learning?", docs)
for r in results:
    print(f"{r['doc']}: {r['score']:.3f}")

## Resources & Next Steps

- **Detailed Explanation**: See `llm/concepts/embeddings.md`
- **Interview Questions**: Review Q&A in markdown file
- **Real-World Examples**: See companies section in markdown
- **Experiment**: Modify the code above and run cells

### Concepts to explore next:
- Related concepts (see markdown file)
- Cross-concept combinations
- Integration with other techniques